# Sample random rows from any split/corpus + run model predictions

Uses the split configs in `configs/splits/*.yaml`. On first use for a given config, materializes the split manifests under `splits/<config_name>/`; subsequent runs reuse them.

Splits: `train` / `val` / `test`. Corpora: `batch01`, `batch02`, `kazattsd`, `kazemo` (present only if in the config).

In [8]:
import os, sys, json, random
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'configs' / 'splits').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('repo:', REPO_ROOT)

os.environ.setdefault('DATASETS_AUDIO_BACKEND', 'soundfile')
os.environ.setdefault('TORCHCODEC_QUIET', '1')

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

import yaml, torch, numpy as np
from splits.builder import build_splits
from splits.materialize import _materialize_per_corpus
from splits.schema import ALL_SPLITS

repo: /home/fatikh/issai/audio-clf


In [9]:
# Available split configs
cfg_dir = REPO_ROOT / 'configs' / 'splits'
for p in sorted(cfg_dir.glob('*.yaml')):
    print(f"{p.name:40s}  name={yaml.safe_load(p.read_text()).get('name')}")

all_three_aug.yaml                        name=b1_b2_kazattsd_aug_v1
all_three_noaug.yaml                      name=b1_b2_kazattsd_noaug_v1
batch01_only_aug.yaml                     name=batch01_only_aug_v1
batch01_only_noaug.yaml                   name=batch01_only_noaug_v1
batch01_plus_batch02_aug.yaml             name=b1_b2_aug_v1
batch01_plus_batch02_noaug.yaml           name=b1_b2_noaug_v1


In [10]:
# ---- configure split / corpus ----
CONFIG_FILE = 'batch01_plus_batch02_noaug.yaml'
SPLIT       = 'test'                 # 'train' | 'val' | 'test'
CORPUS      = None              # or None for all corpora present in the split
N_SAMPLES   = 20
SEED        = 42                   # int for reproducible sampling
# ----------------------------------

config_path = cfg_dir / CONFIG_FILE
cfg = yaml.safe_load(config_path.read_text())
split_dir = REPO_ROOT / (cfg.get('output_dir') or 'splits') / cfg['name']
if not (split_dir / 'config.yaml').exists():
    print(f'Materializing split manifests at {split_dir} (first-time; may download datasets)...')
    build_splits(config_path)
assert SPLIT in ALL_SPLITS
print('split_dir:', split_dir)

split_dir: /home/fatikh/issai/audio-clf/splits/b1_b2_noaug_v1


In [11]:
# ---- configure model checkpoint ----
CKPT_PATH     = REPO_ROOT / 'models' / 'exp1-hubert-b1-b2_20260421-102709' / 'finetune' / 'best_model_finetuned.pt'
#CKPT_PATH     = REPO_ROOT / 'models' / 'exp1-hubert-b1-b2-kz_20260421-102709' / 'finetune' / 'best_model_finetuned.pt'
#CKPT_PATH     = REPO_ROOT / 'models' / 'exp2-wavlm-b1-b2_20260421-180341' / 'finetune' / 'best_model_finetuned.pt'
#CKPT_PATH     = REPO_ROOT / 'models' / 'exp2-wavlm-b1-b2-kz_20260421-180341' / 'finetune' / 'best_model_finetuned.pt'

#CKPT_PATH     = REPO_ROOT / 'models' / 'exp3-hubert-b1-b2-noaug_20260422-042827' / 'finetune' / 'best_model_finetuned.pt'
#CKPT_PATH     = REPO_ROOT / 'models' / 'exp3-wavlm-b1-b2-noaug_20260422-042827' / 'finetune' / 'best_model_finetuned.pt'


BACKBONE_NAME = 'hubert'              # 'hubert' | 'wavlm'   (not stored in ckpt)
PRETRAINED    = 'facebook/hubert-base-ls960'  # or 'microsoft/wavlm-base-plus'
LABEL_ENCODERS = None                 # Path or None -> auto-search (ckpt dir, parent, parent/..)
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
# ------------------------------------

from multihead.model import MultiTaskBackbone
from utils.config import build_feature_extractor

CKPT_PATH = Path(CKPT_PATH)
assert CKPT_PATH.exists(), f'checkpoint not found: {CKPT_PATH}'

def _find_label_encoders(ckpt_path):
    for d in [ckpt_path.parent, ckpt_path.parent.parent, ckpt_path.parent.parent.parent]:
        p = d / 'label_encoders.json'
        if p.exists():
            return p
    return None

le_path = Path(LABEL_ENCODERS) if LABEL_ENCODERS else _find_label_encoders(CKPT_PATH)
assert le_path and le_path.exists(), 'label_encoders.json not found; set LABEL_ENCODERS manually'
encoders = json.load(open(le_path))
# support both {'age': ...} and {'age_category': ...}
emo_map = encoders['emotion']
gen_map = encoders['gender']
age_map = encoders.get('age') or encoders.get('age_category')
inv = lambda m: {v: k for k, v in m.items()}
emo_inv, gen_inv, age_inv = inv(emo_map), inv(gen_map), inv(age_map)
print('label_encoders:', le_path)
print(' emotion:', list(emo_map))
print(' gender :', list(gen_map))
print(' age    :', list(age_map))

ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
n_emo = ckpt.get('num_emotions', len(emo_map))
n_gen = ckpt.get('num_genders',  len(gen_map))
n_age = ckpt.get('num_ages',     len(age_map))

model = MultiTaskBackbone(
    num_emotions=n_emo, num_genders=n_gen, num_ages=n_age,
    freeze_backbone=False, use_spec_augment=False,
    backbone_name=BACKBONE_NAME, pretrained=PRETRAINED,
).to(DEVICE)
missing, unexpected = model.load_state_dict(ckpt['model_state_dict'], strict=False)
if missing or unexpected:
    print(f'[warn] missing={len(missing)} unexpected={len(unexpected)} keys on load_state_dict')
model.eval()
processor = build_feature_extractor(PRETRAINED)
print(f'loaded {CKPT_PATH.name}  epoch={ckpt.get("epoch")} step={ckpt.get("global_step")} device={DEVICE}')

label_encoders: /home/fatikh/issai/audio-clf/models/exp1-hubert-b1-b2_20260421-102709/finetune/label_encoders.json
 emotion: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised']
 gender : ['F', 'M']
 age    : ['adult', 'child', 'senior', 'young']
loaded best_model_finetuned.pt  epoch=4 step=38380 device=cuda


In [12]:
# Prediction helper
@torch.no_grad()
def predict(audio_array, sampling_rate=16000, topk=3):
    inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors='pt')
    iv = inputs['input_values'].to(DEVICE)
    am = inputs.get('attention_mask')
    if am is not None:
        am = am.to(DEVICE)
    emo_logits, gen_logits, age_logits = model(iv, attention_mask=am)
    def _top(logits, inv_map, k):
        probs = torch.softmax(logits[0].float(), dim=-1).cpu().numpy()
        order = np.argsort(-probs)[:k]
        return [(inv_map[int(i)], float(probs[int(i)])) for i in order]
    return {
        'emotion': _top(emo_logits, emo_inv, topk),
        'gender':  _top(gen_logits, gen_inv, min(topk, len(gen_inv))),
        'age':     _top(age_logits, age_inv, min(topk, len(age_inv))),
    }

In [13]:
# Materialize split, sample rows, show audio + true labels + predictions
from datasets import Audio
from IPython.display import Audio as IPyAudio, display

per_corpus = _materialize_per_corpus(split_dir, SPLIT)
print(f'{SPLIT}: corpora =>', {k: len(v) for k, v in per_corpus.items()})
if CORPUS is None:
    pool = [(n, i) for n, ds in per_corpus.items() for i in range(len(ds))]
else:
    assert CORPUS in per_corpus, f'{CORPUS} not in split {SPLIT} (have {list(per_corpus)})'
    pool = [(CORPUS, i) for i in range(len(per_corpus[CORPUS]))]

rng = random.Random(SEED) if SEED is not None else random
picks = rng.sample(pool, k=min(N_SAMPLES, len(pool)))

decoded = {}
def _d(c):
    if c not in decoded:
        ds = per_corpus[c]
        if 'audio' in ds.column_names:
            ds = ds.cast_column('audio', Audio(decode=True, sampling_rate=16000))
        decoded[c] = ds
    return decoded[c]

LABEL_COLS = ['emotion', 'gender', 'age_category']
for corpus, idx in picks:
    row = _d(corpus)[idx]
    audio = row.get('audio')
    sr = audio['sampling_rate']; arr = audio['array']
    truth = {k: row.get(k) for k in LABEL_COLS if k in row and row.get(k) is not None}
    preds = predict(arr, sampling_rate=sr, topk=3)
    print(f'--- corpus={corpus}  idx={idx}  split={SPLIT}  dur={len(arr)/sr:.2f}s ---')
    print('truth :', truth)
    for task, ranked in preds.items():
        head = ', '.join(f'{lbl}={p:.3f}' for lbl, p in ranked)
        print(f'pred {task:8s}: {head}')
    display(IPyAudio(arr, rate=sr))
    print()

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

test: corpora => {'batch01': 675, 'batch02': 1510}
--- corpus=batch01  idx=456  split=test  dur=2.48s ---
truth : {'emotion': 'surprised', 'gender': 'M', 'age_category': 'child'}
pred emotion : angry=0.311, neutral=0.302, fearful=0.169
pred gender  : M=0.994, F=0.006
pred age     : child=0.790, senior=0.195, young=0.010


...


--- corpus=batch01  idx=108  split=test  dur=4.12s ---
truth : {'emotion': 'neutral', 'gender': 'M', 'age_category': 'child'}
pred emotion : neutral=0.609, happy=0.183, fearful=0.075
pred gender  : M=0.918, F=0.082
pred age     : child=0.726, senior=0.103, adult=0.096



--- corpus=batch02  idx=139  split=test  dur=2.16s ---
truth : {'emotion': 'sad', 'gender': 'F', 'age_category': 'child'}
pred emotion : sad=0.770, happy=0.136, neutral=0.036
pred gender  : F=0.986, M=0.014
pred age     : child=0.423, young=0.287, adult=0.196



--- corpus=batch02  idx=1043  split=test  dur=1.08s ---
truth : {'emotion': 'neutral', 'gender': 'M', 'age_category': 'child'}
pred emotion : sad=0.823, neutral=0.108, fearful=0.025
pred gender  : M=0.978, F=0.022
pred age     : child=0.965, senior=0.015, young=0.011



--- corpus=batch02  idx=227  split=test  dur=8.48s ---
truth : {'emotion': 'happy', 'gender': 'M', 'age_category': 'adult'}
pred emotion : happy=0.288, angry=0.203, sad=0.186
pred gender  : M=0.981, F=0.019
pred age     : young=0.673, adult=0.254, child=0.049



--- corpus=batch02  idx=1164  split=test  dur=3.80s ---
truth : {'emotion': 'neutral', 'gender': 'F', 'age_category': 'adult'}
pred emotion : neutral=0.509, happy=0.369, sad=0.050
pred gender  : F=0.907, M=0.093
pred age     : adult=0.465, young=0.329, senior=0.107


In [14]:
# Reusable helpers (call without re-running earlier cells)
def sample(split='train', corpus=None, n=1, seed=None):
    pc = _materialize_per_corpus(split_dir, split)
    pool = ([(n_, i) for n_, ds in pc.items() for i in range(len(ds))] if corpus is None
            else [(corpus, i) for i in range(len(pc[corpus]))])
    rng_ = random.Random(seed) if seed is not None else random
    picks_ = rng_.sample(pool, k=min(n, len(pool)))
    out = []
    for c, i in picks_:
        ds = pc[c].cast_column('audio', Audio(decode=True, sampling_rate=16000))
        out.append((c, i, ds[i]))
    return out

def predict_row(row, topk=3):
    a = row['audio']
    return predict(a['array'], sampling_rate=a['sampling_rate'], topk=topk)

# Example:
#   rows = sample('test', 'kazemo', n=2)
#   for c, i, r in rows: print(c, i, predict_row(r))